In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Input
import statsmodels.api as sm
from linearmodels.iv import IV2SLS
import seaborn as sns
from itertools import product

2025-05-09 17:57:53.733544: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [535]:
"""
Load data and configs function
"""
# Load  data from file
def load_data(file_path) -> pd.DataFrame:
    """
    Load car data from a CSV, JSON, or Excel file.
    
    Args:
        file_path (str): Path to the file.
        
    Returns:
        pd.DataFrame: Loaded data as a pandas DataFrame.
    """
    if file_path.endswith('.csv'):
        data = pd.read_csv(file_path)
    elif file_path.endswith('.json'):
        data = pd.read_json(file_path)
    elif file_path.endswith('.xlsx'):
        data = pd.read_excel(file_path)
    else:
        raise ValueError("Unsupported file format. Please provide a .csv, .json, or .xlsx file.")
    return data

In [537]:

def process_dates(data) -> pd.DataFrame:
    """
    Process selling dates and extract selling year, selling month and selling week-of-the-year
    """
    df = data.copy()
    # Clean the string (remove the part in parentheses)
    df['saledate'] = df['saledate'].str.replace(r'\s*\(.*?\)', '', regex=True).str.strip()
    # Convert to datetime (handles GMT offsets) - invalid dates will become NaT
    df['saledate'] = pd.to_datetime(df['saledate'], utc=True, errors='coerce')
    # Exclude rows with invalid dates
    nb_invalids = df[df['saledate'].isna()].shape[0]
    if nb_invalids>0:
        print(nb_invalids, 'rows with invalid dates have been dropped')
    df = df[~df['saledate'].isna()]
    #Extract selling year, months and weeks
    df['sellingyear'] = df['saledate'].map(lambda x: x.year)
    df['sellingmonth'] = df['saledate'].map(lambda x: x.month)
    #df['sellingweek'] = df['saledate'].map(lambda x: x.dt.isocalendar().week)
    df['sellingweek'] = df['saledate'].map(lambda x: x.isocalendar().week)
    return df


In [539]:
def define_market(data, market_vars, market_label='market', market_code='marketid', minsize=20) -> pd.DataFrame:
    """
    Build two new column (marketid, market) for the market IDs
    Args:
        market_vars (list of str): the names of columns in data that define the market
        minsize (int): minimum market size to allow. Sales in markets with lower sizes will be dropped
    Returns:
        The dataframe data with two new columns:
            market: a concatenate of values from variables in market_vars
            marketid: numerical code of market
            
    ****Note1: The same function can be used to define products
    ****Note2: process_dates should be run before this function (to keep all products, set minsize=1)
    """    
    df = data.copy()
    df['market'] = df[market_vars[0]].astype('str')
    for var in market_vars[1:]:
        df['market'] = df['market'] + '_' + df[var].astype('str')
    markets_tab = df['market'].value_counts()
    selected_markets = markets_tab.iloc[np.where(markets_tab>=minsize)].index
    nb_droppedmarkets = np.sum(~(markets_tab>=minsize))
    nb_droppedrows = np.sum(~df['market'].isin(selected_markets))
    df = df[df['market'].isin(selected_markets)]
    print(f"\n{nb_droppedmarkets} markets with sizes lower than {minsize} have been dropped; making {nb_droppedrows} dropped rows \n")
    idmarket_mask = dict([(i, selected_markets[i]) for i in range(len(selected_markets))])
    marketid_mask = dict([(selected_markets[i], i) for i in range(len(selected_markets))])
    df['marketid'] = df['market'].map(marketid_mask)
    return df

In [541]:
def regroup_categories(series, replacement_mask=dict(),  min_frequency=10000) -> pd.Series:
    """
    Args:
        series (pd.Series): series of strings in lower cases.
        replace_mask (dictionary): a dictionary that maps some values (to be replaced) to their new values
        minsize (int): minimum frequency per category to allow
                        all series' values with a lower frequency will be regrouped in the 'other'
    """
    s = series.copy()
    s = s.replace(replacement_mask)
    frequencies = s.value_counts()
    popular_values = set(frequencies[frequencies>= min_frequency].index)-{''}
    s = s.map(lambda x: x if x in popular_values else 'other')
    return s

In [543]:
def extract_relevant_variables(data, marketvar='marketid', productvar='make', dropna=True,
                               numerical=[], ordered=[], unordered=[], 
                               ordered_masks=dict(), unordered_masks=dict(), 
                               ordered_min_frequencies=dict(), unordered_min_frequencies=dict()) -> pd.DataFrame:
    """
    Extract and process relevant variables
    Args:
        data (pd.DataFrame): shouldcontaain a 'state' column
        marketvar (str): the column name in data for market
        productvar (str): the column name in data for product 
        numerical (list of str): the names of relevant columns that will be set as numerical 
        ordered (list of str): the names of relevant columns that will be set as ordered categorical
        unordered (list of str): the names of relevant columns tht will be set as unordered categorical
        ordered_masks (dict of dict): the masks to be used in the regroup_categories function

    Note1: The distinction in the type of variables can be usefull for missing value imputation using chained equations
    Note2: There is a particular processing for 'color', 'interior' regarding value '—' 
            (which isn't recongnized as a 'minus' symbol in some computers)
    
    """
    df = data.copy()
    if productvar in numerical+ordered+unordered:
        df = df[[marketvar]+['state']+numerical+ordered+unordered]
    else:
        df = df[[marketvar,productvar]+numerical+ordered+unordered]
    for var in ['color', 'interior']:
        if var in df.columns:
            df[var].replace({'—':np.nan})           
    print('\nNumber of missing per relevant variable in the remaining dataframe \n', np.sum(df.isna(), axis=0))
    print(f"\nA total of {np.sum(np.any(df.isna(), axis=1))} rows with missing data in relevant variables have been dropped \n")
    if dropna:
        df = df.dropna()
    for var in ordered:
        df[var] = df[var].map(lambda x: str(x).lower())
        df[var] = regroup_categories(
            df[var], replacement_mask=ordered_masks[var], min_frequency=ordered_min_frequencies[var]
        )
        df[var] = pd.Categorical(df[var], ordered=True)
    for var in unordered:
        df[var] = df[var].map(lambda x: str(x).lower())
        df[var] = regroup_categories(
            df[var], replacement_mask=unordered_masks[var], min_frequency=unordered_min_frequencies[var]
        )
        df[var] = pd.Categorical(df[var], ordered=False)
    for var in numerical:
        df[var] = df[var].astype('float')
    return df

In [545]:
def reduce_to_full_rank(A, tol=1e-10):
    """Return a full-rank version of matrix A by removing linearly dependent columns."""
    #A must be a 2D-array
    Q, R = np.linalg.qr(A)
    independent = np.abs(np.diag(R)) > tol
    return A[:, independent]

def get_non_collinear_instruments(Z, X_exog = None, tol=1e-10):
    # Regress each instrument on the exogenous variables
    # Keep instruments that are not perfectly explained by exogenous variables

    if np.all(X_exog==None):
        X = np.ones((Z.shape[0], 1)) # other exogenous reduce to constant only
    else:
        X = np.asarray(X_exog)
        X = np.hstack([X, np.ones((X.shape[0], 1))])  # add constant

    keep = []
    NewZ = Z
    for col in Z.columns:
        y = Z[col].values
        Xcol0 = np.hstack([NewZ.drop([col],axis=1), X])  # add other instruments
        Xcol = reduce_to_full_rank(Xcol0)
        beta = np.linalg.lstsq(Xcol, y, rcond=None)[0]
        y_hat = Xcol @ beta
        residual = y - y_hat    
        if np.linalg.norm(residual) > tol:
            keep.append(col)
        else:
            NewZ = NewZ.drop([col],axis=1)

    return Z[keep], keep

In [560]:
def aggregate_data(data, pop_data, marketvar='marketid', productvar= 'make', twodegree_polynomial_instruments=False, 
                   aggfunc='mean', numerical=[], categorical=[], dep=[], endog=[], exog=[]):
    """
    Aggregate the data at (market, product)-level
    Compute market sales, then market shares, then log-market share ratios w.r.t the outside option (not buying a car in our dataset) 
    BLP instruments for price (sum of rival products' characteristics) are also extracted (see Berry, Levingson and Pakes, 1995)
    'state' is needed to get market sizes (proxied by state-varying population sizes of year 2015)
    pop_data is a data with populatioon size by state
    """
    #data=df0.copy(); marketvar='marketid'; productvar= 'make'; twodegree_polynomial_instrument=False
    #aggfunc=np.mean, numerical=['sellingprice','odometer','condition'], ordered=[], 
    #unordered=['year','make','body','color','interior']
    
    #Extract categories for each categorical variable (the most frequent category is excluded)
    Categories = [(var, cat) for var in categorical for cat in data[var].value_counts().index[1:]]

    df = data[[marketvar,productvar,'state']+numerical].copy(); pop_df = pop_data.copy()
    depvars = []; endogvars = []; exogvars = []
    for var in numerical:
        if var in dep:
            depvars = depvars + [var]
        if var in endog:
            endogvars = endogvars + [var]
        if var in exog:
            exogvars = exogvars + [var]
    for var, cat in Categories:
        varcat = f"{var}_{cat}"
        df[varcat] = (data[var] == cat).astype(float)
        if var in dep:
            depvars = depvars + [varcat]
        if var in endog:
            endogvars = endogvars + [varcat]
        if var in exog:
            exogvars = exogvars + [varcat]
    df = df.groupby([marketvar, productvar,'state'], observed=True).agg(aggfunc).reset_index()
    df['sales'] = data.groupby([marketvar, productvar,'state'], observed=True).size().values

    pop_df['population (2015)'] = pop_df['population (2015)'].map(lambda x: x.replace(',','')).astype('float')
    outsideoption_df = df[[marketvar,'state','sales']].groupby(by=[marketvar,'state'],observed=True).sum().reset_index().rename({'sales':'allsales'},axis=1)
    pop_df = pop_df.merge(outsideoption_df,on='state')
    df = pop_df.merge(df, on=[marketvar,'state'])

    depvars = depvars + ['population (2015)','allsales', 'sales']

    #Extract instruments
    if twodegree_polynomial_instrument==False:
        Z = df[[marketvar]+exogvars].groupby([marketvar]).apply(
            lambda x: x.assign(**dict(
                [('Nb_RivalProducts',x.shape[0]-1)]+[(var+'_RivalProducts', x[var].sum()-x[var]) for var in exogvars]
            )), include_groups=False
        ).reset_index(drop=True).drop(exogvars, axis=1)
    if twodegree_polynomial_instrument==True: 
        Z = df[[marketvar]+exogvars].groupby([marketvar]).apply(
            lambda x: x.assign(**dict(
                [('Nb_RivalProducts',x.shape[0]-1)]+[(var+'_RivalProducts', x[var].sum()-x[var]) for var in exogvars]+\
                [(var1+'*'+var2+'_RivalProducts', (x[var1]*x[var2]).sum()-x[var1]*x[var2]) for var1, var2 in list(product(exogvars, repeat=2))]
            )), include_groups=False
        ).reset_index(drop=True).drop(exogvars, axis=1)
    Z, instrvars = get_non_collinear_instruments(Z, df[exogvars])
    #df = sm.add_constant(df.join(Z))
    df = df.join(Z)  
    
    return df, depvars, endogvars, exogvars, instrvars
    

In [562]:
cars_file_path = '/Users/colin/Library/CloudStorage/OneDrive-UniversitedeMontreal/On_Going_Studies/IVADO_Project/car_prices.csv'
population_file_path = '/Users/colin/Library/CloudStorage/OneDrive-UniversitedeMontreal/On_Going_Studies/IVADO_Project/states_populations_yr2015.csv'
df = load_data(cars_file_path); pop_df = load_data(population_file_path)

df = process_dates(df)
df = define_market(df, market_vars=['sellingyear','sellingmonth','state'], market_label='market', market_code='marketid', minsize=20)

df['color'] = df['color'].replace({'—':np.nan})
df['interior'] = df['interior'].replace({'—':np.nan})

make_mask = {
    'gmc truck':'gmc', 'dodge tk':'dodge', 'mazda tk':'mazda', 'hyundai tk':'hyundai', 'mercedes-b':'mercedes-benz',
    'chev truck':'chevrolet', 'ford tk':'ford', 'ford truck':'ford', 'vw':'volkswagen'
}
body_mask = body_type_map = {
    #suvs
    'suv': 'suv',      
    #sedans
    'sedan': 'sedan', 'g sedan': 'sedan', 'elantra coupe': 'sedan',
    #coupes
    'coupe': 'coupe', 'g coupe': 'coupe', 'genesis coupe': 'coupe', 'cts coupe': 'coupe', 'koup': 'coupe', 'cts-v coupe': 'coupe',
    'g37 coupe': 'coupe', 'q60 coupe': 'coupe',
    # convertibles
    'convertible': 'convertible', 'g convertible': 'convertible', 'g37 convertible': 'convertible', 'q60 convertible': 'convertible',
    'beetle convertible': 'convertible', 'granturismo convertible': 'convertible',
    # wagons
    'wagon': 'wagon', 'cts wagon': 'wagon', 'tsx sport wagon': 'wagon', 'cts-v wagon': 'wagon',
    # hatchbacks
    'hatchback': 'hatchback',
    # vans / minivans
    'minivan': 'van', 'van': 'van', 'e-series van': 'van', 'promaster cargo van': 'van', 'ram van': 'van', 'transit van': 'van',
    # pickup trucks - cab variants
    'crew cab': 'pickup', 'double cab': 'pickup', 'crewmax cab': 'pickup', 'access cab': 'pickup', 'king cab': 'pickup',
    'supercrew': 'pickup', 'extended cab': 'pickup', 'supercab': 'pickup', 'regular cab': 'pickup', 'regular-cab': 'pickup',
    'quad cab': 'pickup', 'club cab': 'pickup', 'xtracab': 'pickup', 'mega cab': 'pickup', 'cab plus': 'pickup', 'cab plus 4': 'pickup'
}
year_mask = dict([(str(i), str(1999)) for i in range(1980,2000)]) #To troncate all manufacturing years below 1999
color_mask = dict()
interior_mask = dict()

make_freq = 1; body_freq = 10000; year_freq = 1; color_freq = 10000; interior_freq = 10000;

unordered_masks = dict([('year',year_mask),('make',make_mask),('body',body_mask),('color',color_mask),('interior',interior_mask)])
unordered_minfreq = dict([('year',1),('make',10000),('body',1),('color',10000),('interior',10000)])

df = extract_relevant_variables(
    df, marketvar='marketid', productvar='make', dropna=True, 
    numerical=['sellingprice','odometer','condition'], ordered=[], unordered=['year','make','body','color','interior'],
    ordered_masks=dict(), unordered_masks=unordered_masks,  ordered_min_frequencies=dict(), unordered_min_frequencies=unordered_minfreq
)

df,  depvars, endogvars, exogvars, instrvars = aggregate_data(
    df, pop_df, marketvar='marketid', productvar='make', twodegree_polynomial_instruments=False, 
    aggfunc='mean', numerical=['sellingprice','odometer','condition'], 
    categorical=['year','make','body','color','interior'], dep=[], 
    endog=['sellingprice'], exog=['odometer','condition','year','body','color','interior']
)

38 rows with invalid dates have been dropped

63 markets with sizes lower than 20 have been dropped; making 485 dropped rows 


Number of missing per relevant variable in the remaining dataframe 
 marketid            0
state               0
sellingprice        0
odometer           93
condition       11794
year                0
make            10282
body            13174
color           25416
interior        17808
dtype: int64

A total of 60818 rows with missing data in relevant variables have been dropped 



In [564]:
depvars

['population (2015)', 'allsales', 'sales']

In [566]:
endogvars

['sellingprice']

In [570]:
np.array(exogvars)

array(['odometer', 'condition', 'year_2013', 'year_2014', 'year_2011',
       'year_2008', 'year_2007', 'year_2010', 'year_2006', 'year_2009',
       'year_2005', 'year_2004', 'year_2003', 'year_2015', 'year_2002',
       'year_1999', 'year_2001', 'year_2000', 'body_suv', 'body_pickup',
       'body_van', 'body_hatchback', 'body_coupe', 'body_wagon',
       'body_convertible', 'color_white', 'color_gray', 'color_silver',
       'color_blue', 'color_red', 'color_other', 'color_gold',
       'color_green', 'interior_gray', 'interior_beige', 'interior_tan',
       'interior_other'], dtype='<U16')

In [572]:
np.array(instrvars)

array(['Nb_RivalProducts', 'odometer_RivalProducts',
       'condition_RivalProducts', 'year_2013_RivalProducts',
       'year_2014_RivalProducts', 'year_2011_RivalProducts',
       'year_2008_RivalProducts', 'year_2007_RivalProducts',
       'year_2010_RivalProducts', 'year_2006_RivalProducts',
       'year_2009_RivalProducts', 'year_2005_RivalProducts',
       'year_2004_RivalProducts', 'year_2003_RivalProducts',
       'year_2015_RivalProducts', 'year_2002_RivalProducts',
       'year_1999_RivalProducts', 'year_2001_RivalProducts',
       'year_2000_RivalProducts', 'body_suv_RivalProducts',
       'body_pickup_RivalProducts', 'body_van_RivalProducts',
       'body_hatchback_RivalProducts', 'body_coupe_RivalProducts',
       'body_wagon_RivalProducts', 'body_convertible_RivalProducts',
       'color_white_RivalProducts', 'color_gray_RivalProducts',
       'color_silver_RivalProducts', 'color_blue_RivalProducts',
       'color_red_RivalProducts', 'color_other_RivalProducts',
       'c

In [574]:
df

,state,"state, full name",population (2015),marketid,allsales,make,sellingprice,odometer,condition,year_2013,...,color_silver_RivalProducts,color_blue_RivalProducts,color_red_RivalProducts,color_other_RivalProducts,color_gold_RivalProducts,color_green_RivalProducts,interior_gray_RivalProducts,interior_beige_RivalProducts,interior_tan_RivalProducts,interior_other_RivalProducts
0,ca,California (US),39144818.0,1,19533,bmw,20901.973208,66845.083273,33.713251,0.083997,...,2.359580,1.379342,1.230934,0.848863,0.403178,0.299384,4.498197,2.316830,1.090180,0.353838
1,ca,California (US),39144818.0,1,19533,chevrolet,11546.932616,85526.761608,27.988675,0.193092,...,2.301496,1.406730,1.143933,0.820490,0.381559,0.299497,4.233706,2.419035,1.199667,0.449101
2,ca,California (US),39144818.0,1,19533,chrysler,8861.980440,78628.926650,25.496333,0.166259,...,2.353351,1.377189,1.148637,0.806325,0.347401,0.301830,4.360799,2.390792,1.148410,0.462615
3,ca,California (US),39144818.0,1,19533,dodge,10424.014778,80645.782020,25.652709,0.205665,...,2.322279,1.359655,1.130512,0.828838,0.367170,0.305375,4.392501,2.428285,1.199268,0.461742
4,ca,California (US),39144818.0,1,19533,ford,12776.851017,73411.733457,30.821072,0.224399,...,2.356376,1.405048,1.163570,0.830793,0.395911,0.279563,4.231676,2.359021,1.135526,0.440468
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3513,ns,Nova Scotia (Canada),943000.0,217,31,dodge,4825.000000,100984.250000,19.750000,0.000000,...,1.666667,1.333333,0.000000,0.333333,0.000000,0.000000,3.583333,0.000000,1.583333,0.000000
3514,ns,Nova Scotia (Canada),943000.0,217,31,ford,6400.000000,143594.750000,25.500000,0.000000,...,1.666667,1.333333,0.000000,0.333333,0.000000,0.000000,2.583333,0.000000,2.583333,0.000000
3515,ns,Nova Scotia (Canada),943000.0,217,31,hyundai,8216.666667,75747.750000,34.750000,0.000000,...,1.666667,1.333333,0.000000,0.333333,0.000000,0.000000,3.583333,0.000000,1.583333,0.000000
3516,ns,Nova Scotia (Canada),943000.0,217,31,nissan,8900.000000,121700.000000,36.500000,0.000000,...,1.000000,1.000000,0.000000,0.333333,0.000000,0.000000,3.250000,0.000000,2.583333,0.000000
